### Feature Extraction

### import libraries

In [1]:
%pip install gensim

Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding
from gensim.models import Word2Vec

import warnings
warnings.filterwarnings("ignore")


I0000 00:00:1787553102.113357  443836 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1787553102.176830  443836 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1787553103.660623  443836 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


### load the datasets and tokenizer

In [3]:
X_train_padded = np.load("../models/X_train_padded.npy")
X_val_padded = np.load("../models/X_val_padded.npy")
X_test_padded = np.load("../models/X_test_padded.npy")

y_train = np.load("../models/y_train.npy")
y_val = np.load("../models/y_val.npy")
y_test = np.load("../models/y_test.npy")

In [4]:
with open("../models/tokenizer.pkl", "rb") as file:
    tokenizer = pickle.load(file)

In [5]:
print("Training:", X_train_padded.shape)
print("Validation:", X_val_padded.shape)
print("Testing:", X_test_padded.shape)

Training: (34705, 200)
Validation: (7439, 200)
Testing: (7438, 200)


#### Tokenization + Integer Encoding

In [6]:
word_index = tokenizer.word_index
print("Vocabulary size:", len(word_index))

Vocabulary size: 85872


In [7]:
list(word_index.items())[:20]

[('<OOV>', 1),
 ('movie', 2),
 ('film', 3),
 ('not', 4),
 ('one', 5),
 ('like', 6),
 ('good', 7),
 ('no', 8),
 ('time', 9),
 ('even', 10),
 ('would', 11),
 ('story', 12),
 ('really', 13),
 ('see', 14),
 ('well', 15),
 ('much', 16),
 ('bad', 17),
 ('get', 18),
 ('great', 19),
 ('people', 20)]

In [8]:
print(X_train_padded[0])

[    4    16    51     5   432   130   144   291     1   826    27    96
    18    31    64     5  7914  1119  3151   206   193  4363  3441  1181
  1078  2168   743 18546     4    34    16   474     1  1624     1   441
  1110   187  1616    21  2389  4363     1   204   575     1     1 14458
     1  1031 13719   107   552  1119     1   634 12228     1 12457  7291
  4363  9450   351  4363  5739  3343     1   203   815  2422   932     4
  4582   342  2757     1  3521     1  2028   213   686   994  3686  2168
    66  1486     1     1   716     1  1477  1119  2045   983    21   454
  2722   206  1678  7819    50    41   416   120  1591 11470    89  1503
   187   971   897  1294  6254  2455  6254  1961  2288    62     1    11
    16    47  1882   245   374   635   142     1     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0   

#### TF-IDF Representation

In [9]:
X_train_text = pd.read_pickle("../models/X_train_text.pkl")
X_val_text = pd.read_pickle("../models/X_val_text.pkl")
X_test_text = pd.read_pickle("../models/X_test_text.pkl")

In [10]:
print(X_train_text[0])

one reviewers mentioned watching oz episode hooked right exactly happened first thing struck oz brutality unflinching scenes violence set right word go trust not show faint hearted timid show pulls no punches regards drugs sex violence hardcore classic use word called oz nickname given oswald maximum security state penitentary focuses mainly emerald city experimental section prison cells glass fronts face inwards privacy not high agenda em city home many aryans muslims gangstas latinos christians italians irish scuffles death stares dodgy dealings shady agreements never far away would say main appeal show due fact goes shows dare forget pretty pictures painted mainstream audiences forget charm forget romance oz mess around first episode ever saw struck nasty surreal say ready watched developed taste oz got accustomed high levels graphic violence not violence injustice crooked guards sold nickel inmates kill order get away well mannered middle class inmates turned prison bitches due lac

In [11]:
tfidf = TfidfVectorizer(max_features=20000)
#only fit on training data to avoid data leakage
X_train_tfidf = tfidf.fit_transform(X_train_text)
#Transform validation data
X_val_tfidf = tfidf.transform(X_val_text)
#Transform test data
X_test_tfidf = tfidf.transform(X_test_text)

In [12]:
print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Validation TF-IDF shape:", X_val_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

Training TF-IDF shape: (34705, 20000)
Validation TF-IDF shape: (7439, 20000)
Testing TF-IDF shape: (7438, 20000)


### Inspect TF-IDF Features

In [13]:
tfidf_features = tfidf.get_feature_names_out()
print("Number of TF-IDF features:", len(tfidf_features))
print("First 20 TF-IDF features:", tfidf_features[:20])

Number of TF-IDF features: 20000
First 20 TF-IDF features: ['aaa' 'aaliyah' 'aamir' 'aardman' 'aaron' 'ab' 'abandon' 'abandoned'
 'abandoning' 'abandonment' 'abandons' 'abba' 'abbas' 'abbey' 'abbie'
 'abbot' 'abbott' 'abby' 'abc' 'abducted']


In [14]:
print(X_train_tfidf[0])

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 95 stored elements and shape (1, 20000)>
  Coords	Values
  (0, 12170)	0.0720071393500638
  (0, 11744)	0.10779049259305688
  (0, 15463)	0.04360176528360087
  (0, 12403)	0.04936575234079112
  (0, 6172)	0.06638607140066521
  (0, 13757)	0.052476533688610806
  (0, 19794)	0.05433059657652801
  (0, 5549)	0.0618587014410461
  (0, 19986)	0.08651961601875556
  (0, 11729)	0.04008384490346645
  (0, 11082)	0.05113713227129379
  (0, 7428)	0.036657055338587506
  (0, 19419)	0.03995121917384449
  (0, 705)	0.046647087660860086
  (0, 14897)	0.12060560306240684
  (0, 19987)	0.27380926218409873
  (0, 11610)	0.0987675442337045
  (0, 16911)	0.11856520432525118
  (0, 9615)	0.059877275586019754
  (0, 2631)	0.4428402588303912
  (0, 9248)	0.10030948079791864
  (0, 2302)	0.08249646453576248
  (0, 892)	0.08280840757365306
  (0, 15805)	0.18642677599787386
  (0, 17970)	0.07400315413880698
  :	:
  (0, 4004)	0.1005979082319032
  (0, 1903)	0.0901786318484150

### Word Embeddings

In [15]:
sentences = [
    review.split() 
    for review in X_train_text
]

In [16]:
word2vec_model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4
)

In [17]:
word2vec_model.wv["movie"]

array([ 0.13309997,  0.59778595,  0.7340745 ,  0.46516997, -0.53601944,
       -1.4508722 , -0.8598245 ,  2.678121  , -0.88718003, -0.06000064,
       -0.0383819 ,  1.1216478 ,  0.2581944 ,  0.5128846 , -0.08140045,
       -0.90338653,  0.4167732 ,  0.7239243 , -1.2945238 , -2.7416413 ,
        0.55472934, -0.46811387,  2.0534148 ,  0.30472314,  0.38946772,
       -0.68218577,  0.45686552,  1.0990602 ,  1.8033471 ,  1.7796191 ,
       -0.5235428 , -1.6391398 ,  2.0935905 , -1.3973013 , -0.7825584 ,
        1.1148869 ,  1.1212968 , -0.27480656, -1.9221286 ,  1.6984067 ,
        0.8967804 , -0.15906824, -0.73620045,  0.5867633 ,  0.09363513,
       -1.8058516 ,  2.6150594 , -0.6764331 ,  1.2013906 ,  0.3905698 ,
        0.45319927,  0.47211033,  1.5886533 ,  1.7669735 ,  1.0215765 ,
       -0.56937   ,  0.61530733, -1.6104577 , -0.287688  ,  0.17917646,
        1.1482004 , -1.2373914 ,  1.0314007 , -1.6507826 ,  0.7939387 ,
        0.21701336,  2.5861826 ,  3.0585926 , -1.190765  ,  1.20

In [18]:
print(word2vec_model.wv["movie"].shape)

(100,)


### Check Word Similarity

In [19]:
word2vec_model.wv.most_similar("movie", topn=10)

[('film', 0.7866309881210327),
 ('flick', 0.6685290932655334),
 ('movies', 0.6345303654670715),
 ('suppose', 0.6047537326812744),
 ('thats', 0.5979516506195068),
 ('sequel', 0.5920596718788147),
 ('guess', 0.5662664175033569),
 ('figured', 0.5573659539222717),
 ('disappointed', 0.5498062372207642),
 ('sure', 0.5429974794387817)]

### Trainable Embedding Layer

In [20]:
vocab_size = min(20000,len(tokenizer.word_index) + 1)
embedding_dim = 128
max_length = X_train_padded.shape[1]

In [21]:
embedding_layer = Embedding(
    input_dim=vocab_size,
    output_dim=embedding_dim,
    input_length=max_length,
    name="trainable_embedding"
)

In [22]:
# embedding output
#take one paddesreview as input to the embedding layer
sample_input = X_train_padded[:1]
#Pass the sample input through the embedding layer
sample_embedding = embedding_layer(sample_input)
print(sample_embedding.shape)

E0000 00:00:1787553122.810054  443836 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


(1, 200, 128)


### Compare the representations

In [23]:
representation_summary = pd.DataFrame({
    "Representation": [
        "Integer Encoding",
        "TF-IDF",
        "Word2Vec Embedding",
        "Trainable Embedding"
    ],
    "Type": [
        "Sparse integer IDs",
        "Sparse numerical features",
        "Dense word vectors",
        "Dense trainable word vectors"
    ]
})

representation_summary

,Representation,Type
0,Integer Encoding,Sparse integer IDs
1,TF-IDF,Sparse numerical features
2,Word2Vec Embedding,Dense word vectors
3,Trainable Embedding,Dense trainable word vectors


In [24]:
print("Integer Encoding:", X_train_padded.shape)
print("TF-IDF:", X_train_tfidf.shape)
print("Word2Vec dimension:", word2vec_model.wv.vector_size)
print("Trainable Embedding dimension:", embedding_dim)

Integer Encoding: (34705, 200)
TF-IDF: (34705, 20000)
Word2Vec dimension: 100
Trainable Embedding dimension: 128


In [25]:
with open("../models/tfidf_vectorizer.pkl", "wb") as file:
    pickle.dump(tfidf, file)

In [26]:
from scipy.sparse import save_npz

save_npz("../models/X_train_tfidf.npz", X_train_tfidf)
save_npz("../models/X_val_tfidf.npz", X_val_tfidf)
save_npz("../models/X_test_tfidf.npz", X_test_tfidf)

In [27]:
word2vec_model.save("../models/word2vec.model")